# Task 3 — MSA & Semiglobal Alignment
Prepare sequences for an online MSA, analyze conserved motifs, and prototype a semiglobal alignment variant.

## 1. Initialize Project Environment
Import libraries for FASTA handling, motif scoring, and semiglobal alignment sanity checks.

In [31]:
from __future__ import annotations

import logging
import time
import urllib.parse
import urllib.request
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from Bio import AlignIO, SeqIO, pairwise2
from Bio.SeqRecord import SeqRecord

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_data_root() -> Path:
    here = Path().resolve()
    for base in [here, *here.parents]:
        candidate = base / "data/work/AndreiCod/lab01"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/work/AndreiCod/lab01 relative to current directory"
    )


DATA_ROOT = locate_data_root()
FASTAS = list(DATA_ROOT.glob("*.fa*"))
logging.info("FASTA sources: %s", FASTAS)
assert FASTAS, "Need Lab 1 sequences for MSA prep."

[INFO] FASTA sources: [PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/nm000546.fa'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_dna_multi.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_protein_multi.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_mm_protein.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_hs_protein_P04637.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_dr_protein.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_hs_transcript_NM_000546.6.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_dr_transcript_NM_131327.2.fasta'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/my_tp53.fa'), PosixPath('/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_mm_transc

## 2. Define Configuration Parameters
Centralize which sequences feed the MSA, how many to include, and semiglobal scoring options.

In [32]:
@dataclass
class MSAConfig:
    fasta_path: Path
    email: str = "student@example.com"
    max_sequences: int = 5
    export_dir: Path = Path("artifacts")
    semiglobal_settings: Dict[str, float] = None
    msa_trim_bp: Optional[int] = 1500  # Trim for Clustal Omega API limits
    semiglobal_trim_bp: Optional[int] = 2000
    ebi_api_url: str = "https://www.ebi.ac.uk/Tools/services/rest/clustalo"
    poll_interval: int = 5  # seconds between status checks

    def __post_init__(self):
        if self.semiglobal_settings is None:
            self.semiglobal_settings = {
                "match": 2,
                "mismatch": -1,
                "gap_open": -2,
                "gap_extend": 0,
                "penalize_end_gaps": (False, False),
            }

    def describe(self):
        info = asdict(self)
        info["fasta_path"] = str(info["fasta_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = MSAConfig(fasta_path=DATA_ROOT / "my_tp53.fa")
CONFIG.describe()

{'fasta_path': '/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/my_tp53.fa',
 'email': 'student@example.com',
 'max_sequences': 5,
 'export_dir': 'artifacts',
 'semiglobal_settings': {'match': 2,
  'mismatch': -1,
  'gap_open': -2,
  'gap_extend': 0,
  'penalize_end_gaps': (False, False)},
 'msa_trim_bp': 1500,
 'semiglobal_trim_bp': 2000,
 'ebi_api_url': 'https://www.ebi.ac.uk/Tools/services/rest/clustalo',
 'poll_interval': 5}

In [33]:
def trim_record(record: SeqRecord, max_bp: Optional[int]) -> SeqRecord:
    if max_bp is None or len(record.seq) <= max_bp:
        return record
    trimmed = record[:max_bp]
    trimmed.description = f"{record.description} [trimmed_to_{max_bp}]"
    return trimmed


def trim_records(records: List[SeqRecord], max_bp: Optional[int]) -> List[SeqRecord]:
    return [trim_record(rec, max_bp) for rec in records]


In [34]:
def choose_sequences(cfg: MSAConfig) -> List[SeqIO.SeqRecord]:
    selected: List[SeqIO.SeqRecord] = []
    for record in SeqIO.parse(cfg.fasta_path, "fasta"):
        selected.append(record)
        if len(selected) >= cfg.max_sequences:
            break
    if len(selected) < 3:
        raise ValueError("Need at least 3 sequences for Task 3")
    return selected


selected_records = choose_sequences(CONFIG)
len(selected_records)


3

In [35]:
msa_records = trim_records(selected_records, CONFIG.msa_trim_bp)
semiglobal_inputs = trim_records(selected_records[:2], CONFIG.semiglobal_trim_bp)
logging.info(
    "Prepared %d records for MSA (<=%s bp) and %d for semiglobal alignment",
    len(msa_records),
    CONFIG.msa_trim_bp or "all",
    len(semiglobal_inputs),
)


[INFO] Prepared 3 records for MSA (<=1500 bp) and 2 for semiglobal alignment


## 3. Implement Core Functionality
Submit sequences to Clustal Omega via EBI REST API, parse returned alignments, mark conserved motifs, and run a semiglobal demo using Biopython.

In [36]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
subset_path = EXPORT_DIR / "task3_subset_for_msa.fasta"
SeqIO.write(msa_records, subset_path, "fasta")
subset_path


PosixPath('artifacts/task3_subset_for_msa.fasta')

In [37]:
from io import StringIO


def submit_clustalo_job(sequences: List[SeqRecord], cfg: MSAConfig) -> str:
    """Submit sequences to EBI Clustal Omega REST API and return job ID."""
    fasta_io = StringIO()
    SeqIO.write(sequences, fasta_io, "fasta")
    fasta_string = fasta_io.getvalue()

    params = {
        "email": cfg.email,
        "sequence": fasta_string,
        "outfmt": "clustal_num",
    }

    data = urllib.parse.urlencode(params).encode("utf-8")
    url = f"{cfg.ebi_api_url}/run"

    logging.info("Submitting %d sequences to Clustal Omega API...", len(sequences))
    req = urllib.request.Request(url, data=data, method="POST")
    with urllib.request.urlopen(req, timeout=60) as response:
        job_id = response.read().decode("utf-8").strip()

    logging.info("Job submitted: %s", job_id)
    return job_id


def poll_job_status(job_id: str, cfg: MSAConfig, max_wait: int = 300) -> str:
    """Poll EBI API until job completes. Returns final status."""
    url = f"{cfg.ebi_api_url}/status/{job_id}"
    elapsed = 0

    while elapsed < max_wait:
        with urllib.request.urlopen(url, timeout=30) as response:
            status = response.read().decode("utf-8").strip()

        if status == "FINISHED":
            logging.info("Job %s finished successfully", job_id)
            return status
        elif status in ("FAILURE", "ERROR"):
            raise RuntimeError(f"Clustal Omega job failed: {status}")

        logging.info("Job status: %s (waiting %ds...)", status, cfg.poll_interval)
        time.sleep(cfg.poll_interval)
        elapsed += cfg.poll_interval

    raise TimeoutError(f"Job {job_id} did not complete within {max_wait}s")


def fetch_alignment_result(job_id: str, cfg: MSAConfig) -> str:
    """Fetch alignment result from completed job."""
    url = f"{cfg.ebi_api_url}/result/{job_id}/aln-clustal_num"
    with urllib.request.urlopen(url, timeout=60) as response:
        result = response.read().decode("utf-8")
    return result


# Submit job to Clustal Omega API and wait for result
msa_result_path = EXPORT_DIR / "task3_msa_result.clustal"

job_id = submit_clustalo_job(msa_records, CONFIG)
poll_job_status(job_id, CONFIG)
alignment_text = fetch_alignment_result(job_id, CONFIG)

# Save alignment
with open(msa_result_path, "w") as f:
    f.write(alignment_text)

print(f"[OK] MSA result saved to: {msa_result_path}")
print(f"\nFirst 1000 characters of alignment:\n{alignment_text[:1000]}")

[INFO] Submitting 3 sequences to Clustal Omega API...
[INFO] Job submitted: clustalo-R20251230-115958-0859-49780359-p1m
[INFO] Job clustalo-R20251230-115958-0859-49780359-p1m finished successfully


[OK] MSA result saved to: artifacts/task3_msa_result.clustal

First 1000 characters of alignment:
CLUSTAL O(1.2.4) multiple sequence alignment


NM_000546.6      --------------------------------------CTCAAAAGTCTAGAGCCACCGT	22
NM_011640.3      TTTCCCCTCCCACGTGCTCACCCTGGCTAAAGTTCTGTAGCTTCAGTTCATTGGGACCAT	60
NM_131327.2      ----------------------------------CTGTAACTAGG---------------	11
                                                         *                   

NM_000546.6      CCAGGGAGCAGGTAGCTGCTGGGCTCCGGGGACACTTTGCGTTCGGGCTGGGAGCGTGCT	82
NM_011640.3      CCTGGCTGTAGGTAGCGACTACAGTTAGGGGGCACCTAGCATTCAGGCCCTCATCCTCC-	119
NM_131327.2      ----------GGAATCCCCAAAACTCCAC-GCGGATTTGCTTTGTGGATGTCCAATA---	57
                           ** * *  *     *     *     * ** **  **             

NM_000546.6      TTCCACGACGGTGACACGCTTCCCTGGATTGGCAGCCAGACTGCCTTCCGGGTCACTGCC	142
NM_011640.3      -------------TCCTTCCCAGCAGGGTGTCACGCTTCTCCGAAGACTGGATGACTGCC	166
NM_131327.2      -------------ACCTCCTTGTTTTGG-

In [38]:
def load_alignment(path: Path):
    alignment = AlignIO.read(path, "clustal")
    logging.info(
        "Loaded alignment with %d sequences, length %d",
        len(alignment),
        alignment.get_alignment_length(),
    )
    return alignment


if msa_result_path.exists():
    alignment = load_alignment(msa_result_path)
    alignment

[INFO] Loaded alignment with 3 sequences, length 1614


In [39]:
def conservation_scores(alignment, threshold: float = 0.9) -> pd.DataFrame:
    columns = []
    aln_len = alignment.get_alignment_length()
    for pos in range(aln_len):
        column = alignment[:, pos]
        counts = pd.Series(list(column)).value_counts()
        top_base = counts.index[0]
        score = counts.iloc[0] / counts.sum()
        columns.append({"position": pos, "top_base": top_base, "score": score})
    df = pd.DataFrame(columns)
    df["conserved"] = df["score"] >= threshold
    return df


if "alignment" in locals():
    conservation_df = conservation_scores(alignment)
    conservation_df.head()

In [40]:
if "alignment" in locals():
    conserved_blocks = conservation_df[conservation_df["conserved"]]
    if conserved_blocks.empty:
        logging.warning(
            "No conserved positions reached the threshold; consider lowering it."
        )
    else:
        conserved_blocks.head()


In [41]:
from Bio.Align import MultipleSeqAlignment
from io import StringIO

if (
    "alignment" in locals()
    and "conserved_blocks" in locals()
    and not conserved_blocks.empty
):
    start_pos = int(conserved_blocks.iloc[0]["position"])
    end_pos = start_pos + 20
    conserved_excerpt = alignment[:, start_pos:end_pos]

    # Format as FASTA manually since format() was removed in newer Biopython
    excerpt_fasta = StringIO()
    AlignIO.write(conserved_excerpt, excerpt_fasta, "fasta")
    print("Conserved excerpt (FASTA):")
    print(excerpt_fasta.getvalue())

Conserved excerpt (FASTA):
>NM_000546.6
CAAAAGTCTAGAGCCACCGT
>NM_011640.3
CTTCAGTTCATTGGGACCAT
>NM_131327.2
CTAGG---------------



In [42]:
def run_semiglobal(seq1, seq2, cfg: MSAConfig):
    params = cfg.semiglobal_settings
    alignment = pairwise2.align.globalms(
        seq1.seq,
        seq2.seq,
        params["match"],
        params["mismatch"],
        params["gap_open"],
        params["gap_extend"],
        penalize_end_gaps=params["penalize_end_gaps"],
        one_alignment_only=True,
    )
    return alignment[0]


if len(semiglobal_inputs) < 2:
    raise ValueError("Need at least two sequences for semiglobal alignment")

semiglobal_alignment = run_semiglobal(
    semiglobal_inputs[0], semiglobal_inputs[1], CONFIG
)
print(semiglobal_alignment[2])
print(semiglobal_alignment[0][:120])
print(semiglobal_alignment[1][:120])


2257.0
----------------CTCA--------AAAGT-CTAG-AGC--CACCGTCCA--GGGA--------GC---AGGTAGCTGCTGGGCTCC-----GGGGACACTTTGCGTTCGGG-----
TTTCCCCTCCCACGTGCTCACCCTGGCTAAAGTTCT-GTAGCTTCA--GTTCATTGGGACCATCCTGGCTGTAGGTAGC-----GACTACAGTTAGGGGGCACCTAGCATTCAGGCCCTC


## 4. Validate with Unit Tests
Sanity-check the conservation scoring and semiglobal helper on synthetic data to avoid surprises.

In [43]:
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


def test_conservation_scores():
    dummy_alignment = MultipleSeqAlignment(
        [
            SeqRecord(Seq("AAAA"), id="s1"),
            SeqRecord(Seq("AAAT"), id="s2"),
            SeqRecord(Seq("AAAC"), id="s3"),
        ]
    )
    df = conservation_scores(dummy_alignment, threshold=0.66)
    assert df.loc[df["position"] == 0, "conserved"].iloc[0]


def test_semiglobal_alignment():
    seq1 = SeqRecord(Seq("TTTAAA"), id="a")
    seq2 = SeqRecord(Seq("AAA"), id="b")
    aln = run_semiglobal(seq1, seq2, CONFIG)
    assert "AAA" in aln[0]


if "alignment" not in locals():
    test_conservation_scores()
    logging.info("Conservation scoring tests passed on dummy data.")

test_semiglobal_alignment()
logging.info("Semiglobal helper test passed.")

[INFO] Semiglobal helper test passed.


## 5. Analyze Performance Metrics
Record runtime of the conservation computation and semiglobal alignment to estimate scaling behavior.

In [44]:
import time

perf_rows = []
if "alignment" in locals():
    start = time.perf_counter()
    conservation_scores(alignment)
    perf_rows.append({"task": "conservation", "runtime_s": time.perf_counter() - start})

if len(semiglobal_inputs) >= 2:
    start = time.perf_counter()
    run_semiglobal(semiglobal_inputs[0], semiglobal_inputs[1], CONFIG)
    perf_rows.append({"task": "semiglobal", "runtime_s": time.perf_counter() - start})
else:
    logging.warning("Skipping semiglobal benchmark; not enough trimmed sequences.")

pd.DataFrame(perf_rows)


,task,runtime_s
0,conservation,0.212396
1,semiglobal,0.260164


In [45]:
if (
    "alignment" in locals()
    and "conserved_blocks" in locals()
    and not conserved_blocks.empty
):
    conserved_blocks.to_csv(EXPORT_DIR / "task3_conserved_blocks.csv", index=False)
    if "conserved_excerpt" in locals():
        from Bio import AlignIO
        from io import StringIO

        sio = StringIO()
        AlignIO.write(conserved_excerpt, sio, "fasta")
        with open(EXPORT_DIR / "task3_conserved_excerpt.fasta", "w") as fh:
            fh.write(sio.getvalue())

with open(EXPORT_DIR / "task3_semiglobal_alignment.txt", "w") as fh:
    seq1, seq2, score, start, end = semiglobal_alignment
    fh.write(f"Score: {score}\nStart:{start} End:{end}\n")
    fh.write(seq1 + "\n" + seq2 + "\n")

print("Artifacts saved to", EXPORT_DIR)

Artifacts saved to artifacts
